In [0]:
# =============================================================================
# Notebook  : 02b_map_03_crm_events
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_03_crm_events
# Spec Ref  : §1.3 CRM Events Mapping (Salesforce Task → Standardized Events)
# Purpose   : Read from hgi.silver.tasks WHERE source = 'salesforce' AND object = 'Task'
#             Classify Subject field → meta_event vocabulary
#             Output: hgi.silver.crm_events
#
# Key spec requirement:
#   meta_event must be 100% non-null (fallback = cleaned subject string)
#   contact_id built from WhoId/WhatId already resolved in bronze layer
#
# Inputs  : hgi.silver.tasks  (written by 02a_silver_task)
# Output  : hgi.silver.crm_events
#
# Run after: map_01 (contacts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/FINAL_pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

print(f"=== Map 03: crm_events (SF Tasks → standardized events) ===  customer={customer_id}")
print(f"  Input  : {sv}.tasks")
print(f"  Output : {sv}.crm_events")

In [0]:
# CELL 3 ── Create crm_events table if not exists
create_map_table(
    f"{sv}.crm_events",
    """
        id              STRING NOT NULL,
        tenant          BIGINT,
        event_timestamp TIMESTAMP,
        contact_id      STRING,
        meta_event      STRING,
        event_text      STRING
    """,
    cluster_by="tenant, meta_event"
)

In [0]:
# CELL 4 ── Build crm_events with full meta_event classification
# Spec §1.3: CASE WHEN subject patterns → meta_event
# Spec requirement: meta_event must be 100% non-null
#   The ELSE clause normalises raw subject to lowercase_underscore
#   so it is NEVER null — 'unknown_task' is the absolute fallback.

# Safe drop in case target exists as a VIEW
safe_drop_view(f"{sv}.crm_events")

spark.sql(f"""
    CREATE OR REPLACE TABLE {sv}.crm_events
    USING DELTA
    CLUSTER BY (tenant, meta_event)
    {DELTA_TBLPROPS_MAP}
    AS
    SELECT
        id,
        tenant,
        event_timestamp,

        -- Composite contact_id from WhoId/WhatId (already resolved in bronze layer)
        -- Format: salesforce_Contact_Id_003... or salesforce_Lead_Id_00Q...
        CONCAT(
            'salesforce_',
            COALESCE(contact_source_system_object, 'Contact'),
            '_Id_',
            contact_source_key_value
        ) AS contact_id,

        -- meta_event classification per spec §1.3
        -- 100% non-null: ELSE branch cleans the raw subject → never returns NULL
        CASE
            WHEN LOWER(a_subject) = 'call'                          THEN 'call'
            WHEN LOWER(a_subject) LIKE 'email: re:%'                THEN 'email_reply'
            WHEN LOWER(a_subject) LIKE 'email:%'                    THEN 'email_sent'
            WHEN LOWER(a_subject) LIKE 'meeting:%'                  THEN 'meeting'
            WHEN LOWER(a_subject) LIKE 'demo:%'                     THEN 'demo'
            WHEN LOWER(a_subject) LIKE 'follow%up%'                 THEN 'follow_up'
            WHEN LOWER(a_subject) LIKE 'call:%'                     THEN 'call'
            WHEN LOWER(a_subject) LIKE 'webinar%'                   THEN 'webinar'
            WHEN LOWER(a_subject) LIKE 'linkedin%'                  THEN 'linkedin_outreach'
            WHEN LOWER(a_subject) LIKE 'signup%'                    THEN 'signup'
            WHEN LOWER(a_subject) LIKE '%sign up%'                  THEN 'signup'
            WHEN LOWER(a_subject) LIKE '%conversion%'               THEN 'conversion'
            WHEN LOWER(a_subject) LIKE '%closed won%'               THEN 'closed_won'
            WHEN LOWER(a_subject) LIKE '%closed lost%'              THEN 'closed_lost'
            WHEN a_subject IS NULL                                   THEN 'unknown_task'
            ELSE
                -- Normalise raw subject: lowercase, spaces/specials → underscore
                -- This ensures 100% non-null meta_event per spec gate
                COALESCE(
                    LOWER(REGEXP_REPLACE(a_subject, '[^a-zA-Z0-9]+', '_')),
                    'unknown_task'
                )
        END AS meta_event,

        a_subject AS event_text

    FROM {sv}.tasks
    WHERE source_system     = 'salesforce'
      AND source_system_object = 'Task'
      AND contact_source_key_value IS NOT NULL
""")


In [0]:
# CELL 5 ── Spec DQ verification
total     = cnt(f"{sv}.crm_events")
null_meta = spark.sql(
    f"SELECT COUNT(*) FROM {sv}.crm_events WHERE meta_event IS NULL"
).collect()[0][0]
null_contact = spark.sql(
    f"SELECT COUNT(*) FROM {sv}.crm_events WHERE contact_id IS NULL"
).collect()[0][0]

print(f"  crm_events: {total:,} rows")
print(f"  null meta_event : {null_meta}   (spec gate = 0, target = 100% non-null)")
print(f"  null contact_id : {null_contact}  (these were filtered by contact_source_key_value IS NOT NULL)")

# Show distribution of meta_event types for visibility
print("\n  meta_event distribution:")
spark.sql(f"""
    SELECT meta_event, COUNT(*) AS cnt
    FROM {sv}.crm_events
    GROUP BY meta_event
    ORDER BY cnt DESC
    LIMIT 10
""").show(truncate=False)

dbutils.notebook.exit("Success")
